# Week 3 Day 5 Lab 1 (free route) - The Copilot on Render

## Running the Copilot beside the app and the agent

On the AWS route, Lab 1 deploys all of UnderCover again, with the Copilot on AgentCore Runtime beside the claim agent and an AgentCore Memory of its own. On the free route we run the Copilot as a third web service on Render, beside your app and the agent from Day 4, with the context layer inside the same service.

AgentCore Runtime, the Gateway and the Memory are AWS services, so for those you can watch me. The rest of today's build works the same way: the Copilot acts with the sign-in of the handler who asked, shows each tool call in the sidebar as it happens, asks before it pays and keeps each conversation in Supabase.

This notebook has no code to run. It gives you the steps, with links to Render's own documentation.

## 5 steps:

STEP 1: Update the app and the agent on Render

STEP 2: Create the Copilot's web service

STEP 3: Point the app at the Copilot

STEP 4: The handover conversation

STEP 5: Compare with the AWS route

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This is the free version of today's first lab. The main lab, <a href="week3day5_lab1.ipynb">week3day5_lab1.ipynb</a>, runs on AWS. You need the two Render services from Day 4, and today's build of the product, yours or <code>week3day5-end</code>, pushed to GitHub as on Day 2. Step 2 needs one name from your build, and shows you where to find it.<br/><br/>I do this week on AWS, so I'm not recording the Render steps. Render's website changes from time to time, so if a screen looks different from what's described here, follow the linked page in Render's documentation.<br/><br/>Step 4 works two claims with the agent and asks the Copilot five questions, which can use most of your 50 free requests for the day. Start it on a day you haven't used any. If you run out, the requests come back the next day.
            </span>
        </td>
    </tr>
</table>

## Step 1: Update the app and the agent on Render

Today's build adds the Copilot's routes to the app, and the Copilot's code to the `agent` package beside the claim agent's. So both of your services on Render need today's code.

- **To deploy my build,** open each web service in the [Render Dashboard](https://dashboard.render.com), go to **Settings**, change the **Branch** to `week3day5-end`, and choose **Save changes**. Render starts the deploy straight away. The branch list shows only the first 15 branches, so type `week3day5-end` to find it.
- **To deploy your own build,** push your branch to your own GitHub repository, as on Day 2. If it's a new branch, change each service's **Branch** to it under **Settings**. Then choose **Manual Deploy** and **Deploy latest commit** on each service.

The build and start commands and the environment variables stay as they were on Day 4. Step 3 adds one more to the app.

Render's documentation: [Deploys](https://render.com/docs/deploys).

## Step 2: Create the Copilot's web service

### Find the Copilot's app

Render starts the Copilot with uvicorn, as it starts the agent, so it needs to know where the Copilot's AgentCore app is, written as `module:name`. The claim agent's is `agent.entrypoint:app`.

Your build chose where the Copilot's app lives, and `package.json` shows it. The script that `npm run dev` uses to serve the Copilot on port 8081 runs `uvicorn` with the Copilot's `module:name`, as `dev:agent` runs `uvicorn agent.entrypoint:app`. If your build starts the Copilot some other way, ask Claude Code which `module:name` uvicorn needs to serve the Copilot's AgentCore app.

To check it, run this in a terminal in the `undercover` folder, with your `module:name` in place of `<copilot-app>`. It installs only the packages Render will, so a missing one shows up here:

```
uv run --isolated --only-group runtime --only-group context uvicorn <copilot-app> --port 8090
```

It should start without errors, and `http://127.0.0.1:8090/ping` in your browser answers with `"status": "Healthy"`. Stop it with Ctrl+C.

### Create the service

In the [Render Dashboard](https://dashboard.render.com), choose **New**, then **Web Service**, and pick the same repository and branch as your app. If the branch isn't in the list, type its name. Then fill in the form, with your `module:name` in place of `<copilot-app>`:

| Field | Value |
|---|---|
| Name | `undercover-copilot` |
| Language | Python 3 |
| Region | The same as your app |
| Build Command | `uv sync --frozen --only-group runtime --only-group context` |
| Start Command | `uv run --no-sync python -m context & uv run --no-sync uvicorn <copilot-app> --host 0.0.0.0 --port $PORT` |
| Compute | $0 / month |

As on Day 4, replace any Build Command Render fills in, and under **Compute** choose **$0 / month**, since Render selects $7 / month again.

Under **Advanced**, set the **Health Check Path** to `/ping`.

These are the agent's commands from Day 4, with the Copilot's app in place of the agent's. The Copilot's packages are in the `runtime` group with the claim agent's, since on the AWS route both runtimes deploy from one package. The context layer runs in the background inside the service, at `http://127.0.0.1:8006/mcp`, where the Copilot looks for it by default, as the agent does in its own service.

Under **Environment Variables**, add these lines:

| Key | Value |
|---|---|
| `OPENROUTER_API_KEY` | From your `.env` |
| `DATABASE_URL` | The same session pooler address as in your `.env` |
| `UNDERCOVER_API` | Your app's address on Render, such as `https://undercover.onrender.com` |

The Copilot doesn't get the agent's key, and unlike the agent it doesn't ask for one, so anyone can reach its address. The handler's sign-in is what protects it. Everything the Copilot does carries the sign-in of the handler who asked, and the app checks it on every call. Each turn starts by asking the app who is signed in, and a message without a valid sign-in ends there, before any call to the model.

With no memory setting, the Copilot's memory is off, as the claim agent's is on the free route.

Choose **Deploy web service**, and when the deploy is live, `https://<your-copilot>.onrender.com/ping` answers with `"status": "Healthy"`.

Render's documentation: [Web services](https://render.com/docs/web-services), [Environment variables and secrets](https://render.com/docs/configure-environment-variables), [Health checks](https://render.com/docs/health-checks).

## Step 3: Point the app at the Copilot

Open your app's web service, and under **Environment**, add:

| Key | Value |
|---|---|
| `UNDERCOVER_COPILOT` | Your Copilot's address on Render, such as `https://undercover-copilot.onrender.com` |

Choose **Edit**, then **Add variable**, and save with **Save and deploy**. It's under the arrow beside **Save, rebuild, and deploy**, which also works but rebuilds the app first. The app now sends each message to the Copilot on Render, with the handler's sign-in, as it does to the Copilot on your laptop.

A free service spins down after 15 minutes without a request, and the first request after that takes about a minute. So before you start a conversation, open the `/ping` addresses of the Copilot and the agent in your browser and wait until both answer. If you leave the conversation for more than 15 minutes, open both again before you carry on.

Render's documentation: [Free instances](https://render.com/docs/free).

## Step 4: The handover conversation

On the AWS route, the conversation starts with a dispatch of the whole queue, which needs more requests than a day's free allowance. On the free route we work two of the approvals over the limit with the agent on your laptop, then ask the Copilot about them. In a terminal in the `undercover` folder, reseed and start everything:

```
npm run seed
npm run dev
```

In a second terminal, work two claims:

```
uv run -m agent CLM-1003
uv run -m agent CLM-1011
```

If the agent approves a claim, it waits for sign-off. If it declines or refers one of them instead, work CLM-1008 as well. Then stop `npm run dev` with Ctrl+C. The rest happens on Render, and Supabase's free plan allows only 15 database connections at once.

Open your app on Render and sign in as Dana Whitfield, who can sign off over the limit. The Copilot's sidebar is open on the right. In one conversation, ask it these four questions, one at a time:

1. **"What needs me?"** The Copilot looks up the queue and names the payouts waiting for a sign-off.
2. **"Why does CLM-1011 need me?"**, or another claim it named. Watch the cards for what the Copilot looks up, such as the claim, the agent's steps on it and lookups in the context layer. A free model may look up less.
3. **"Send it back with a note"**, saying what to look at again. On CLM-1011, that might be the $350 Meridian Air has already paid, or the rule that a pair or a set, such as the suit, counts as one item. The app sends it back as Dana and asks the agent on Render to resume the claim, which carries on from where your laptop left it in Supabase.
4. **"Sign off on another one."** The Copilot picks another payout waiting for a sign-off, and its graph pauses before it pays, with the confirmation card in the sidebar. Press Sign off and pay. The app sends Dana's answer to the Copilot, which resumes the conversation from its checkpoint and signs off as Dana.

Then restart the Copilot: on its **Deploys** page in the Render Dashboard, choose **Manual Deploy** and **Restart service**. When it's back, reload the dashboard, open the conversation from **Earlier conversations**, the history icon at the top of the sidebar, and ask a follow-up, such as "Which claims have we dealt with?". The conversation's state is in Supabase, so the Copilot carries on where it left off.

The main lab's fifth question, "Are we meeting the success criteria we agreed?", runs an evaluation of the whole held-out set, which also needs more than the free allowance. If you have credit on OpenRouter, you can have the main lab's whole conversation instead, starting with a dispatch on the agent's service.

## Step 5: Compare with the AWS route

On the AWS route, each conversation has a Runtime session in a microVM of its own. If the handler answers the confirmation card after that session has ended, the Copilot picks up its pause in a new one. On Render the Copilot is one process that stays up between requests and sleeps after 15 minutes without one. Both keep the conversation in Supabase, which is why the restart lost nothing.

The AWS route also gives the Copilot an AgentCore Memory with a namespace for each handler, so it remembers what each handler has told it from one conversation to the next. Watch Part 6 of the main lab for the memory.

### When you've finished

Your three services on Render cost nothing, so you can keep them as long as you like. When you no longer need them, delete each one from its **Settings** page in the Render Dashboard.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">Build your own LangGraph solution for a business problem where agents make decisions within limits, like approving a claim. Give it decisions with limits, where a person signs off above the limit; durable state, with LangGraph checkpoints in a database; an enterprise context layer for the agent; a memory with a namespace for each user, which on the free route can be LangGraph's <code>PostgresStore</code> in your Supabase database, with pgvector to search it; and deploy it, on Render on the free route. It can be a variation of UnderCover or something completely different.<br/><br/>I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>

In [1]:
# Local verification of the Render (free route) deployment.
import os
import time
from datetime import datetime, timezone

import requests
from dotenv import load_dotenv

load_dotenv("../.env", override=True)
APP = os.getenv("UNDERCOVER_API", "https://undercover-ap.onrender.com").rstrip("/")
print("Checked at (UTC):", datetime.now(timezone.utc).isoformat(timespec="seconds"))
print("App:", APP)


def check(path):
    started = time.time()
    try:
        response = requests.get(APP + path, timeout=120)
        body = response.text.strip().replace("\n", " ")[:160]
        return f"{path}: HTTP {response.status_code} in {time.time() - started:.1f}s - {body}"
    except Exception as exc:
        return f"{path}: {type(exc).__name__}: {str(exc)[:160]}"


for path in ["/api/health", "/sign-in", "/api/claims/CLM-1901"]:
    print(check(path))

for name in ["UNDERCOVER_AGENT", "UNDERCOVER_COPILOT"]:
    print(f"{name} setting:", os.getenv(name, "not set locally (set in Render dashboard)"))


Checked at (UTC): 2026-11-27T05:31:50+00:00
App: https://undercover-ap.onrender.com


/api/health: HTTP 200 in 0.6s - {"status":"ok"}


/sign-in: HTTP 200 in 0.5s - <!doctype html> <html lang="en">   <head>     <meta charset="UTF-8" />     <meta name="viewport" content="width=device-width, initial-scale=1.0" />     <title>U


/api/claims/CLM-1901: HTTP 401 in 0.6s - {"detail":"Sign in, or send the agent's API key"}
UNDERCOVER_AGENT setting: not set locally (set in Render dashboard)
UNDERCOVER_COPILOT setting: not set locally (set in Render dashboard)


## Solution - assignment challenge

### A LangGraph solution for expense reimbursements on the free route

This is the design I would submit; it is a proposal, not a report of a system I have built or run.

**Business problem.** At Beacon Labs (an illustrative company), employees submit expense reports with receipts. A finance assistant checks each report against the travel policy, the approved-vendor list and the employee's earlier reports, then approves a reimbursement, declines it or refers it. About 120 reports a week wait two days on average, at a loaded cost of roughly $18 each. An agent can do the first pass: decide `approve`, `decline` or `refer`, work out the approved amount from the policy's rules, pay amounts up to **$250** itself, and hold everything above that for a finance manager's sign-off. Decisions are recorded with a reason, a policy section, the maker and the policy version. A manager can also send a held decision back with a note of up to 1,000 characters; the agent then decides again, and refers when the policy does not support the note.

**Limits and state rules.** The $250 limit lives in a settings row seeded from `data/company.json`, never in a prompt. The payment operation refuses any amount over the limit in force *before* it reads the report, so an over-limit payment gets the same answer on every report. Only a manager may sign off or send back; the agent's key never stands in for a manager. Each report has one thread and an attempt counter; a new attempt gets a new thread, while carrying on after a stop keeps the same thread and attempt. An evaluation runs on copies, sees no memory and is never signed off.

### Architecture

```
Browser / dashboard
   |  signed-in handler (Supabase Auth)
   v
expenses-app  (Render free web service: FastAPI + built React dashboard)
   |  creates runs, owns locks/heartbeats, validates records and payments
   |  POST /invocations with the agent key (or boto3 on the AWS route)
   v
expenses-agent (Render free web service: BedrockAgentCoreApp)
   |  dispatcher -> per-report LangGraph graphs -> tools
   |  direct route: app API (agent key) + context MCP on 127.0.0.1:8006
   |  memory: LangGraph PostgresStore, namespace ("employees", employee_id)
   v
Supabase Postgres: business tables + LangGraph checkpoints + pgvector store
```

The context layer runs as a background process inside the agent service on the free route, as in the labs. I would keep the option of putting an AgentCore Gateway in front later; the code chooses a route when a run's agents are built, so the graph does not change.

### Modules

| Where | Module | Job |
| --- | --- | --- |
| App | `backend/main.py` | FastAPI app, `health`, handlers, dashboard mount |
| App | `backend/database.py` | SQLAlchemy Core tables in an `expenses` schema, Supabase session pooler or SQLite in tests, `now()` |
| App | `backend/dependencies.py`, `backend/auth.py`, `backend/api_key.py` | One gate per route: signed-in handler, manager, or agent key compared with `compare_digest`; Supabase JWT checked with PyJWT |
| App | `backend/reports.py` | Report CRUD: `start_report`, `record_step`, `record_decision`, `issue_reimbursement`, `hold_payment`, `sign_off`, `send_back`, `take_report`, heartbeats, model calls, work time |
| App | `backend/limits.py`, `backend/calculation.py`, `backend/records.py` | Refuse over-limit amounts first; keep the calculation for the attempt; check a decision record (reason, section, maker, payout, manual version) |
| App | `backend/runs.py`, `backend/work_lock.py` | Create runs and their report copies, one piece of work at a time, mark five-minute-quiet work stopped, finish with a summary built from records, no model |
| App | `backend/agent_service.py` | One function per agent operation, the only place with the agent's address and `/invocations` |
| App | `backend/seed.py` | Seed from `data/` and `context/`; when `MODEL_NAME` is set, make it version 1 and the only model allowed |
| Agent | `agent/connection.py`, `agent/api.py` | The one HTTP client and one function per app operation |
| Agent | `agent/routes.py`, `agent/direct.py`, `agent/gateway.py`, `agent/mcp_tools.py` | One route interface; direct uses the app API and the context MCP server; the Gateway option is added later, all MCP calls go through `mcp_tools.py` |
| Agent | `agent/tools.py` | Report tools and context lookups; no tool takes an id from the model or pays a report |
| Agent | `agent/screening.py` | Hidden-instruction check on a separate model, once per thread, with Structured Outputs and a retry policy |
| Agent | `agent/graph.py`, `agent/harness.py` | Nodes and conditional edges; builds one graph per report with its own saver and route; `work_report`, `carry_on`, `work_in_run`, `resume` |
| Agent | `agent/dispatcher.py` | Model-free `StateGraph` using `Send`, `max_concurrency` and `RetryPolicy` |
| Agent | `agent/memory.py` | `PostgresStore` setup, save and search in an employee's namespace |
| Agent | `agent/handler_answer.py`, `agent/heartbeat.py` | Manager's answer as a message written in code; 60-second heartbeat |
| Agent | `agent/entrypoint.py`, `agent/work.py` | AgentCore app with a discriminated-union payload, background tasks, readiness on first request; run and resume work with heartbeats |
| Agent | `agent/tracing.py` | Langfuse wrapper; without keys it logs a warning and sends nothing |
| Context | `context/files.py`, `context/server.py` | Read the seven owned files once, split documents into numbered sections, serve read-only FastMCP tools with sources |
| Dashboard | `dashboard/src/api.ts`, `auth.ts`, pages `Queue.tsx`, `Report.tsx`, `Runs.tsx`, `Settings.tsx` | Refresh every two seconds; show steps, sources, cost, reason and sign-off cards |

### LangGraph graph

The state holds messages, the screening result, the recorded decision and status, the manager's answer and a model-call counter:

```python
class ReportState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    screening: dict[str, Any] | None
    decision: dict[str, Any] | None
    status: str | None
    manager_answer: dict[str, Any] | None
    model_calls: int

class ReportContext(BaseModel):
    report_id: str
    employee_id: str
    run_id: str | None
    attempt: int
    hold: bool          # False in evaluations
```

Nodes, in order: `screen` (security check, first step on the thread); `refer` (a flagged report is referred to a human and the model never sees it); `agent` (the model, with a tool-binding and a call limit, recording tokens and `usage.cost` after each call); `tools` (`ToolNode` with `handle_tool_errors=False`, so refusals reach the graph); `record` (the app validates and stores the decision); `issue` (the graph pays an approval through the route, never through a model tool); `wait_for_manager` (`interrupt()` for an over-limit approval or a refused payment); `apply_review` (clears the stale decision, restores the model allowance and adds the manager's note as a human message); `remember` (saves the report and decision to the employee's store); and `finish`.

```python
builder = StateGraph(ReportState, context_schema=ReportContext)
builder.add_node("screen", screen_claim)
builder.add_node("refer", refer)
builder.add_node("agent", call_model)
builder.add_node("tools", ToolNode(tools, handle_tool_errors=False))
builder.add_node("record", record_decision)
builder.add_node("issue", issue_reimbursement)
builder.add_node("wait_for_manager", wait_for_manager)
builder.add_node("apply_review", apply_review)
builder.add_node("remember", remember)

builder.add_edge(START, "screen")
builder.add_conditional_edges(
    "screen",
    lambda state: "refer" if state["screening"]["instructions_found"] else "agent",
    {"refer": "refer", "agent": "agent"},
)
builder.add_conditional_edges(
    "agent",
    lambda state: "tools" if state["messages"][-1].tool_calls else "record",
    {"tools": "tools", "record": "record"},
)
builder.add_edge("tools", "agent")
builder.add_conditional_edges(
    "record",
    route_after_decision,
    {
        "issue": "issue",
        "wait_for_manager": "wait_for_manager",
        "remember": "remember",
    },
)
builder.add_conditional_edges(
    "issue",
    lambda state: "wait_for_manager" if state["status"] == "waiting_for_sign_off" else "remember",
    {"wait_for_manager": "wait_for_manager", "remember": "remember"},
)
builder.add_edge("wait_for_manager", "apply_review")
builder.add_edge("apply_review", "agent")
builder.add_edge("remember", END)

graph = builder.compile(checkpointer=saver, store=store)
result = graph.invoke(
    initial_state,
    config={"configurable": {"thread_id": thread_id}, "context": report_context},
    durability="sync",
)
```

`route_after_decision` sends declines and referrals straight to `remember`, approvals within the limit to `issue`, and over-limit approvals to `wait_for_manager`. In an evaluation the graph has no pause: a held approval ends the run and is scored as finished.

The manager's answer is read from the app's record, not from the model:

```python
def wait_for_manager(state: ReportState) -> dict[str, Any]:
    answer = interrupt({
        "kind": "sign_off",
        "report_id": state["report_id"],
        "amount": state["decision"]["amount"],
    })
    return {"manager_answer": answer}

# on sign-off or send-back, on the same thread:
graph.invoke(Command(resume=answer_of(report_id)), config=config, durability="sync")
```

A sign-off has already paid the report in the app before the resume, so `apply_review` ends the graph. A send-back adds a message written in code that names the manager, says the earlier decision no longer stands and nothing was paid, and asks the agent to treat the note as a senior review checked against the policy and to refer the report if the policy does not support it.

### Checkpointer, store and memory

A `PostgresSaver` of its own is built for each report on the process's small shared connection pool, because a saver lets one report at a time read or write. `setup()` is run once at deploy time. `durability="sync"` means every step is on disk before the next one starts, which is what makes a restart carry on from the last saved step.

```python
from langgraph.checkpoint.postgres import PostgresSaver

with PostgresSaver.from_conn_string(DATABASE_URL) as saver:
    saver.setup()
```

Memory is a `PostgresStore` in the same Supabase database, also set up once. Every item lives under `("employees", employee_id)`, so one employee's memory is never used on another employee's report. `store.search` with a query uses pgvector; `setup()` creates the extension and the index, and I would also enable `vector` in the Supabase dashboard. Embeddings point at a model the key supports; for unit tests I would use `FakeEmbeddings`.

```python
from langchain_openai import OpenAIEmbeddings
from langgraph.store.postgres import PostgresStore

index = {
    "dims": 1536,
    "embed": OpenAIEmbeddings(
        model="openai/text-embedding-3-small",
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
    ),
    "fields": ["summary"],
}

with PostgresStore.from_conn_string(DATABASE_URL, index=index) as store:
    store.setup()
    store.put(
        ("employees", employee_id),
        f"report:{report_id}",
        {"summary": summary, "decision": decision, "amount": amount, "section": section},
    )
    hits = store.search(("employees", employee_id), query=question, limit=5)
```

Only a report the agent decided is saved, never the raw story or attachments, and nothing is saved for a report the security check referred. Evaluations get no store.

### Routes

- **HTTP routes in the app:** `GET /api/health`, `GET /api/reports`, `GET /api/reports/{id}`, `GET /api/me`; `POST /api/reports/{id}/start`, `/steps`, `/decision`, `/reimbursement`, `/hold`, `/heartbeat`; `POST /api/reports/{id}/sign-off` and `/send-back` (manager only); `POST /api/runs`, `POST /api/runs/{id}/heartbeat`, `/claims`, `/finish`, `/failure`; `GET /api/runs/{id}` for the agent and `GET /api/runs/latest` for the dashboard. Every operation has an `operation_id`, a one-line docstring and no optional inputs.
- **Graph routes:** `START → screen → {refer, agent}`; `agent ↔ tools` while there are tool calls; `record → {issue, wait_for_manager, remember}`; `issue → {wait_for_manager, remember}`; `wait_for_manager → apply_review → agent`; `remember → END`. The dispatcher is `START → list → fan_out (Send, max_concurrency N) → finish → END`, with a `RetryPolicy` on the work node.
- **Tool routes:** `agent/direct.py` on the free route (app API with the key, context MCP at `http://127.0.0.1:8006/mcp`); `agent/gateway.py` if a Gateway URL is set. The route is chosen when a run's agents are built, and a refused call comes back to the model as a refusal rather than an exception.
- **Service routes on Render:** the browser only reaches the app; the app reaches the agent at `EXPENSES_AGENT` with the key; the agent reaches the app at `EXPENSES_API` with the same key and the context layer inside its own service. Only the app and the agent are exposed on the internet.

### Observability

Every step is a row in `steps` with a kind (`screening`, `tool_call`, `tool_result`, `lookup`, `memory_search`, `memory_save`, `refusal`, `error`, `manager_review`) and, for lookups, the files it came from with their system, owner and review date. `model_calls` holds input and output tokens and the cost from OpenRouter's `usage.cost`; `work_times` holds each spell of work; run summaries and evaluation scores are built from those records with no model. The dashboard refreshes every two seconds so a handler watches the report being worked. Each graph run is also a Langfuse trace in a session named after the report and thread, tagged with the run id; if the free route has no Langfuse keys, the wrapper logs one warning and tracing is off. Render's dashboard carries the deploy logs and health-check history. That combination gives a live view, a cost view and an audit view without depending on one vendor.

### Running it on Render (free route)

1. In Supabase, keep the Week 2 project; enable the `vector` extension and use the session pooler connection string. Set `MODEL_NAME` (for example `openrouter/free`) in `.env` before seeding so version 1 and the allowed models use the free model.
2. Create `expenses-app`: Python 3, $0 compute, region closest to Supabase, health path `/api/health`, branch pushed to GitHub, build command `uv sync --frozen --only-group server && npm ci --ignore-scripts && npm run build`, start command `uv run --no-sync uvicorn backend.main:app --host 0.0.0.0 --port $PORT`. Environment: `DATABASE_URL`, `VITE_SUPABASE_URL`, `VITE_SUPABASE_PUBLISHABLE_KEY`, `EXPENSES_API_KEY`, `MODEL_NAME`. Secrets are set in the Render dashboard, not committed.
3. Create `expenses-agent`: Python 3, $0 compute, same region and branch, health path `/ping`, build command `uv sync --frozen --only-group runtime --only-group context`, start command `uv run --no-sync python -m context & uv run --no-sync uvicorn agent.entrypoint:app --host 0.0.0.0 --port $PORT`. Environment: `OPENROUTER_API_KEY`, `EXPENSES_API_KEY` (the same key), `DATABASE_URL`, `EXPENSES_API` (the app's `onrender.com` address). The context MCP server listens on `127.0.0.1:8006` inside this service.
4. Add `EXPENSES_AGENT` (the agent's address) to the app's environment and save with **Save and deploy**; the app waits ten seconds for the agent, so open `/ping` first to wake the free service.
5. Locally, `npm run seed` then `npm run dev` runs app, agent, context layer and dashboard; `uv run -m agent EXP-1001` works one report from the command line. Sign in as the seeded manager to sign off an over-limit report.
6. Checks I would run before calling it done: stub-model unit tests for every rule (limit, record validation, send-back, refusal, resume); a one-report live run; a restart mid-run to confirm the same attempt carries on from its checkpoint; a batch run at N=3 and an evaluation of the tiny labelled set; and a check of the steps, sources, cost and time in the dashboard. These are checks to perform, not results I am claiming.

**What I would leave out of the first version:** supplier emails, changes to the vendor master, masking of personal fields for the agent, and an AgentCore Memory on the AWS route; the PostgresStore namespace is enough for the pilot.